In [0]:
from pyspark.sql import functions as F

# TODO:
# 1. Load NOAA AIS source
# 2. Inspect source schema
# 3. Map source columns to canonical schema
# 4. Add audit metadata
# 5. Write to bronze_ais_noaa

# Canonical target schema:
# vessel_id
# event_timestamp
# latitude
# longitude
# speed_knots
# heading
# source_system
# ingestion_timestamp
# record_id

# source_df = spark.read....
# mapped_df = ...
# bronze_df = ...
# bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_ais_noaa")

In [0]:
from pyspark.sql import functions as F

sample_noaa_df = spark.createDataFrame(
    [
        ("367123456", "2025-01-01 10:00:00", 37.95, 23.63, 12.4, 180.0),
        ("367123457", "2025-01-01 10:05:00", 37.96, 23.64, 8.2, 175.0),
    ],
    ["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG"]
)

display(sample_noaa_df)

In [0]:
mapped_noaa_df = (
    sample_noaa_df
    .withColumnRenamed("MMSI", "vessel_id")
    .withColumnRenamed("BaseDateTime", "event_timestamp")
    .withColumnRenamed("LAT", "latitude")
    .withColumnRenamed("LON", "longitude")
    .withColumnRenamed("SOG", "speed_knots")
    .withColumnRenamed("COG", "heading")
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("source_system", F.lit("noaa"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_id", F.expr("uuid()"))
)

display(mapped_noaa_df)

In [0]:
mapped_noaa_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_ais_noaa")

In [0]:
spark.table("bronze_ais_noaa").printSchema()
display(spark.table("bronze_ais_noaa"))